# 3D Heterogeneous Seepage SSR (COMSOL Mesh)

This notebook is a structured Jupyter version of `slope_stability_3D_hetero_seepage_SSR_comsol.m`.

It performs:
1. Mesh and quadrature setup
2. Seepage solution
3. Mechanical assembly
4. SSR continuation solve
5. Inline visualization in notebook output

## Notes Before Running

- This workflow is computationally expensive (large 3D model).
- It uses project package folders (`+ASSEMBLY`, `+MESH`, `+SEEPAGE`, `+VIZ`, etc.).
- The default linear solver is `DFGMRES_HYPRE_BOOMERAMG`.
- Inline plotting is enabled for Octave kernel using `%plot -f png`.

In [2]:
%plot -f png
close all; clearvars; clc;

% Load sparsersb for multithreaded sparse matrix-vector products
pkg load sparsersb;

% Configure toolkit for both GUI and headless Jupyter environments.
tks = available_graphics_toolkits();
if any(strcmp(tks, 'qt'))
    graphics_toolkit('qt');
    set(0, 'defaultfigurevisible', 'off');
elseif any(strcmp(tks, 'fltk'))
    graphics_toolkit('fltk');
    % fltk cannot print hidden figures in this environment.
    set(0, 'defaultfigurevisible', 'on');
end

% Ensure we are in the project folder where +package directories are visible.
if exist('ASSEMBLY.quadrature_volume_3D', 'file') ~= 2
    if isfolder('slope_stability')
        cd('slope_stability');
    end
end

fprintf('Working directory: %s\n', pwd);
fprintf('Graphics toolkit: %s\n', graphics_toolkit());



Working directory: /home/beremi/repos/slope_stability/slope_stability
Graphics toolkit: gnuplot


## 1) Main Input Data

Define FE type, material properties, geometry, and mesh density.

In [3]:

% elem_type - type of finite elements; available choices: 'P1', 'P2'
elem_type='P2';

% Davis_type - choice of Davis' approach; available choices: 'A','B','C'
Davis_type='B';

% Mechanical parameters per subdomain:
% [c0, phi, psi, young, poisson, gamma_sat, gamma_unsat]
mat_props = [15, 30,  0, 10000, 0.33, 19, 19;  % Cover layer
    15, 38,  0, 50000, 0.30, 22, 22;  % General foundation
    10, 35,  0, 50000, 0.30, 21, 21;  % Relatively weak foundation
    18, 32,  0, 20000, 0.33, 20, 20;  % General slope mass
    ];

% Hydraulic conductivity for each subdomain [m/s]
k = 1.0;

% Geometrical parameters
x1 = 15;  % length in front of slope
x2 = 10;  % slope length in x
x3 = 15;  % length behind slope
y1 = 10;  % height below slope
y2 = 10;  % slope height
z  = 5;   % length in z direction

% Mesh density parameter
N_h = 1;


## 2) Reference Element Data and Mesh Load

Build quadrature/basis data and load COMSOL mesh.

In [4]:
% quadrature points and weights for volume integration
[Xi, WF] = ASSEMBLY.quadrature_volume_3D(elem_type);

% local basis functions and their derivatives
[HatP, DHatP1, DHatP2, DHatP3] = ASSEMBLY.local_basis_volume_3D(elem_type, Xi);

% Mesh file
file_path = 'meshes/comsol_mesh.h5';
[coord, elem, surf, Q, material, triangle_labels] = MESH.load_mesh_P2(file_path, 1);

% Mesh statistics
n_n = size(coord,2);
n_unknown = length(coord(Q));
n_e = size(elem,2);
n_q = length(WF);
n_int = n_e * n_q;

fprintf('\n');
fprintf('Mesh data:');
fprintf('  number of nodes =%d ', n_n);
fprintf('  number of unknowns =%d ', n_unknown);
fprintf('  number of elements =%d ', n_e);
fprintf('  number of integration points =%d ', n_int);
fprintf('\n');

% Homogeneous identifier for seepage helper call
material_identifier = zeros(1, n_e);


Mesh data:  number of nodes =80226   number of unknowns =231201   number of elements =55895   number of integration points =614845 


## 3) Seepage Problem

Solve pore pressure and saturation state.

In [5]:
% Hydraulic conductivity at each integration point
conduct0 = SEEPAGE.heter_conduct(material_identifier, n_q, k);

% Specific weight of water [kPa]
grho = 9.81;

% Boundary conditions from COMSOL-labeled surfaces
[Q_w, pw_D] = MESH.seepage_boundary_3D_hetero_comsol(coord, surf, triangle_labels, grho);

% Solve seepage
[pw, grad_p, mater_sat] = SEEPAGE.seepage_problem_3D(
    coord, elem, Q_w, pw_D, grho, conduct0, HatP, DHatP1, DHatP2, DHatP3, WF);

% Integration-point saturation mask
mater_sat_ext = repmat(mater_sat, n_q, 1);
saturation = mater_sat_ext(:);

Newton solver converges: number of iteration=10 stopping criterion=4.748891e-17 


## 4) Mechanical Material Fields and Assembly

Map material properties to integration points and assemble elastic stiffness and body-force vector.

In [6]:
fields = {'c0', 'phi', 'psi', 'young', 'poisson', 'gamma_sat', 'gamma_unsat'};
materials = cellfun(@(x) cell2struct(num2cell(x), fields, 2), num2cell(mat_props, 2), 'UniformOutput', false);

[c0, phi, psi, shear, bulk, lame, gamma] = ...
    ASSEMBLY.heterogenous_materials(material, saturation, n_q, materials);

% Elastic stiffness matrix + element-level derivative data (DPhi1/2/3)
[K_elast, B, WEIGHT, DPhi1_out, DPhi2_out, DPhi3_out] = ASSEMBLY.elastic_stiffness_matrix_3D(
    elem, coord, shear, bulk, DHatP1, DHatP2, DHatP3, WF);

% Volume forces at integration points: mechanical-only baseline
f_V_int = [zeros(1, n_int); -gamma; zeros(1, n_int)];
f_V = ASSEMBLY.vector_volume_3D(elem, coord, f_V_int, HatP, WEIGHT);

## 5) Continuation, Newton, and Linear Solver Parameters

In [7]:
% Continuation parameters
lambda_init = 1.0;
d_lambda_init = 0.1;
d_lambda_min = 1e-5;
d_lambda_diff_scaled_min = 0.005;
omega_max_stop = 340700000;
step_max = 100;

% Newton parameters
it_newt_max = 50;
it_damp_max = 10;
tol = 1e-4;
r_min = 1e-4;

% Linear solver settings
agmg_folder = "agmg";
solver_type = 'DFGMRES_HYPRE_BOOMERAMG';

linear_solver_tolerance = 1e-1;
linear_solver_maxit = 100;
deflation_basis_tolerance = 1e-3;
linear_solver_printing = 1;

boomeramg_opts = struct('threads', 16, 'print_level', 0, ...
    'use_as_preconditioner', true);

linear_system_solver = LINEAR_SOLVERS.set_linear_solver(agmg_folder, solver_type, ...
    linear_solver_tolerance, linear_solver_maxit, deflation_basis_tolerance, ...
    linear_solver_printing, Q, coord, boomeramg_opts);

% Constitutive model object
n_strain = 6;
constitutive_matrix_builder = CONSTITUTIVE_PROBLEM.CONSTITUTIVE(
    B, c0, phi, psi, Davis_type, shear, bulk, lame, WEIGHT, n_strain, n_int, 3);

% Provide element geometry for fast element-level tangent assembly (208x speedup).
% This enables the C mex kernel instead of global B'*D*B sparse triple product.
constitutive_matrix_builder.set_element_data(elem, DPhi1_out, DPhi2_out, DPhi3_out);

Element data set: n_p=10, n_e=55895, n_q=11, mex=1


## 6) Run SSR Continuation

By default in this demonstration notebook:
- `direct_on = 0`
- `indirect_on = 1`

In [8]:
direct_on = 0;
indirect_on = 1;

if direct_on
    fprintf('\n Direct continuation method\n');
    tic;
    [U2, lambda_hist2, omega_hist2, Umax_hist2] = CONTINUATION.SSR_direct_continuation(...
        lambda_init, d_lambda_init, d_lambda_min, d_lambda_diff_scaled_min, step_max, ...
        it_newt_max, it_damp_max, tol, r_min, K_elast, Q, f_V, ...
        constitutive_matrix_builder, linear_system_solver.copy());
    time_run = toc;
    fprintf('Running_time = %f \n', time_run);
end

if indirect_on
    fprintf('\n Indirect continuation method\n');
    tic;
    [U3, lambda_hist3, omega_hist3, Umax_hist3, stats] = CONTINUATION.SSR_indirect_continuation(...
        lambda_init, d_lambda_init, d_lambda_min, d_lambda_diff_scaled_min, step_max, ...
        omega_max_stop, it_newt_max, it_damp_max, tol, r_min, K_elast, Q, f_V, ...
        constitutive_matrix_builder, linear_system_solver.copy());
    time_run = toc;
    fprintf('Running_time = %f \n', time_run);
end

if ~isempty(strfind(upper(char(solver_type)), 'BOOMERAMG'))
    LINEAR_SOLVERS.hypre_boomeramg_clear();
end


 Indirect continuation method
Initialising K_r(Q,Q) sparse pattern ... done  (29.9 s, n_Q = 231201, nnz = 18763319)
Building element scatter map ... done  (4.3 s, n_local_dof=30, map_size=402 MB)

 Step = 1  
3|  newton it=1  resid=1.00e+00  alpha=1.00  r=0.0001  step_time=1.46 s
3|  newton it=2  resid=7.92e-02  alpha=1.00  r=0.0001  step_time=1.43 s
3|  newton it=3  resid=3.86e-03  alpha=1.00  r=0.0001  step_time=1.43 s
3|  newton it=4  resid=5.27e-04  alpha=1.00  r=0.0001  step_time=1.42 s
3|  newton it=5  resid=1.04e-04  alpha=1.00  r=0.0001  step_time=1.42 s
Newton method converges: iteration = 6, stopping criterion = 9.596944e-06
   lambda_init = 1, d_lambda_init = 0.1, omega_init = 6013302.6515

 Step = 2  
3|  newton it=1  resid=5.09e-03  alpha=0.98  r=0.0001  step_time=2.54 s
2|  newton it=2  resid=2.56e-03  alpha=1.00  r=0.0001  step_time=1.40 s
3|  newton it=3  resid=1.92e-04  alpha=1.00  r=0.0001  step_time=1.49 s
Newton method converges: iteration = 4, stopping criterion =

## Newton Profiler Summary

Display accumulated timing from `newton_ind_SSR` across all continuation attempts:
- **Top-level** breakdown sorted by total time (descending)
- **Sub-profiles** for `build_F_and_DS_all`, `damping_ALG5`, and `dfgmres_solver` with per-line timing and hit counts

In [9]:
if indirect_on && exist('stats', 'var') && isfield(stats, 'newton_timing')
    nt = stats.newton_timing;

    %% --- Top-level timing table (existing fields only) ---
    top_fields = {'build_F_and_DS', 'solve_V', 'A_orthogonalize', 'damping', ...
                  'setup_preconditioner', 'build_F_eps', 'solve_W', ...
                  'K_r_assembly', 'sparsersb_build', ...
                  'expand_deflation_W', 'expand_deflation_V'};
    top_labels = top_fields(isfield(nt, top_fields));
    top_times  = cellfun(@(f) nt.(f), top_labels);
    total      = sum(top_times);

    [top_times, idx] = sort(top_times, 'descend');
    top_labels = top_labels(idx);
    pcts       = 100 * top_times / total;

    desc_map = struct( ...
        'build_F_and_DS',      'build_F_and_DS_all     (constitutive F + DS)', ...
        'solve_V',             'linear_solver.solve     (V solve)',           ...
        'A_orthogonalize',     'linear_solver.A_orthogonalize',              ...
        'damping',             'NEWTON.damping_ALG5     (line search)',       ...
        'setup_preconditioner','setup/update preconditioner (HYPRE IJV)',     ...
        'build_F_eps',         'build_F_all             (F_eps for diff)',    ...
        'solve_W',             'linear_solver.solve     (W solve)',           ...
        'K_r_assembly',        'K_r sparsersb assembly  (BtDB prebuilt)',    ...
        'expand_deflation_W',  'expand_deflation_basis  (W)',                ...
        'expand_deflation_V',  'expand_deflation_basis  (V)');

    fprintf('\n============================================================\n');
    fprintf('  Newton Profiler  (newton_ind_SSR accumulated)\n');
    fprintf('============================================================\n');
    fprintf('  %-6s  %-8s  %s\n', 'Time', '  %', 'Operation');
    fprintf('  %-6s  %-8s  %s\n', '------', '------', '------------------------------');
    for k = 1:numel(top_labels)
        lbl = top_labels{k};
        if isfield(desc_map, lbl)
            desc = desc_map.(lbl);
        else
            desc = lbl;
        end
        fprintf('  %6.1fs  %5.1f%%   %s\n', top_times(k), pcts(k), desc);
    end
    fprintf('  %-6s  %-8s  %s\n', '------', '------', '------------------------------');
    fprintf('  %6.1fs  %5.1f%%   %s\n', total, 100.0, 'TOTAL');
    fprintf('============================================================\n');

    %% --- Sub-profiling: build_F_and_DS_all breakdown ---
    if isfield(nt, 'build_F_DS__n_calls')
        fprintf('\n  Sub-profile: build_F_and_DS_all  (%d calls)\n', nt.build_F_DS__n_calls);
        fprintf('  %-6s  %-8s  %s\n', '------', '------', '------------------------------');
        sub_items = { ...
            nt.build_F_DS__reduction,      'reduction(lambda)'; ...
            nt.build_F_DS__stress_tangent, 'constitutive_problem_stress_tangent(U)'; ...
            nt.build_F_DS__build_F,        'build_F()'};
        sub_total = nt.build_F_and_DS;
        for k = 1:size(sub_items, 1)
            fprintf('  %6.2fs  %5.1f%%   %s\n', sub_items{k,1}, ...
                100 * sub_items{k,1} / max(sub_total, 1e-12), sub_items{k,2});
        end
    end

    %% --- Sub-profiling: damping_ALG5 breakdown ---
    if isfield(nt, 'damping__n_calls')
        fprintf('\n  Sub-profile: damping_ALG5  (%d calls, %d damp iters total)\n', ...
            nt.damping__n_calls, nt.damping__n_iters);
        fprintf('  %-6s  %-8s  %s\n', '------', '------', '------------------------------');
        sub_items = { ...
            nt.damping__build_F, 'build_F_all  (constitutive evaluation)'; ...
            nt.damping__norm,    'norm(F(Q)-f(Q))  (residual check)'};
        sub_total = nt.damping;
        for k = 1:size(sub_items, 1)
            fprintf('  %6.2fs  %5.1f%%   %s\n', sub_items{k,1}, ...
                100 * sub_items{k,1} / max(sub_total, 1e-12), sub_items{k,2});
        end
    end

    %% --- Sub-profiling: dfgmres_solver breakdown (V+W combined) ---
    if isfield(nt, 'solve__n_calls')
        fprintf('\n  Sub-profile: dfgmres_solver  (%d calls, %d GMRES iters, %d matvecs, %d prec applies)\n', ...
            nt.solve__n_calls, nt.solve__n_gmres_iters, ...
            nt.solve__n_matvecs, nt.solve__n_prec_applies);
        fprintf('  %-6s  %-8s  %s\n', '------', '------', '------------------------------');
        sub_items = { ...
            nt.solve__precond,     'M(v)  preconditioner apply'; ...
            nt.solve__matvec,      'A*w   matrix-vector product'; ...
            nt.solve__project,     'Proj(w) deflation projection'; ...
            nt.solve__ortho,       'Arnoldi orthogonalisation'; ...
            nt.solve__leastsq,     'least-squares + residual check'; ...
            nt.solve__init,        'init (coarse solve + r0)'; ...
            nt.solve__reconstruct, 'solution reconstruction'};
        sub_total = nt.solve_V + nt.solve_W;
        for k = 1:size(sub_items, 1)
            fprintf('  %6.2fs  %5.1f%%   %s\n', sub_items{k,1}, ...
                100 * sub_items{k,1} / max(sub_total, 1e-12), sub_items{k,2});
        end
    end

    fprintf('============================================================\n');
else
    fprintf('No Newton timing data available.\n');
end


  Newton Profiler  (newton_ind_SSR accumulated)
  Time      %       Operation
  ------  ------    ------------------------------
   255.1s   31.1%   linear_solver.solve     (V solve)
   122.2s   14.9%   setup/update preconditioner (HYPRE IJV)
   111.5s   13.6%   NEWTON.damping_ALG5     (line search)
    82.2s   10.0%   linear_solver.A_orthogonalize
    61.9s    7.5%   build_F_and_DS_all     (constitutive F + DS)
    60.9s    7.4%   K_r sparsersb assembly  (BtDB prebuilt)
    51.2s    6.2%   build_F_all             (F_eps for diff)
    49.1s    6.0%   sparsersb_build
    23.9s    2.9%   linear_solver.solve     (W solve)
     1.7s    0.2%   expand_deflation_basis  (W)
     1.7s    0.2%   expand_deflation_basis  (V)
  ------  ------    ------------------------------
   821.3s  100.0%   TOTAL

  Sub-profile: build_F_and_DS_all  (320 calls)
  ------  ------    ------------------------------
   15.82s   25.6%   reduction(lambda)
   33.34s   53.9%   constitutive_problem_stress_tangent(U)
   

In [10]:
linear_system_solver.iteration_collector.iterations

ans =
{
  [1,1] = {}(0x0)
  [1,2] = {}(0x0)
  [1,3] =
  {
    [1,1] = 3
    [1,2] = 3
    [1,3] = 3
    [1,4] = 3
    [1,5] = 3
    [1,6] = 3
    [1,7] = 2
    [1,8] = 3
  }

  [1,4] =
  {
    [1,1] = 3
    [1,2] = 3
    [1,3] = 2
    [1,4] = 3
    [1,5] = 1
    [1,6] = 2
  }

  [1,5] =
  {
    [1,1] = 2
    [1,2] = 2
    [1,3] = 2
    [1,4] = 3
    [1,5] = 1
    [1,6] = 2
  }

  [1,6] =
  {
    [1,1] = 2
    [1,2] = 0
    [1,3] = 2
    [1,4] = 3
    [1,5] = 1
    [1,6] = 2
  }

  [1,7] =
  {
    [1,1] = 1
    [1,2] = 0
    [1,3] = 2
    [1,4] = 3
    [1,5] = 1
    [1,6] = 2
  }

  [1,8] =
  {
    [1,1] = 2
    [1,2] = 1
    [1,3] = 2
    [1,4] = 3
    [1,5] = 1
    [1,6] = 2
  }

  [1,9] =
  {
    [1,1] = 2
    [1,2] = 1
    [1,3] = 3
    [1,4] = 3
    [1,5] = 2
    [1,6] = 2
    [1,7] = 1
    [1,8] = 3
  }

  [1,10] =
  {
    [1,1] = 2
    [1,2] = 1
    [1,3] = 2
    [1,4] = 3
    [1,5] = 1
    [1,6] = 2
  }

  [1,11] =
  {
    [1,1] = 2
    [1,2] = 1
    [1,3] = 3
    [1,4] = 3
    

## 7) Postprocessing and Inline Visualization

This cell reproduces the script visualization and shows figures directly in notebook output.

% Keep only boundary surface faces
v = elem(1:4,:);
F = [v([2 3 4],:), v([1 3 4],:), v([1 2 4],:), v([1 2 3],:)];
Fs = sort(F,1).';
[~,~,ic] = unique(Fs, 'rows');
cnt = accumarray(ic, 1);
bndFs = Fs(cnt(ic) == 1, :);

surfV = surf(1:3,:);
surfKs = sort(surfV,1).';
isBnd = ismember(surfKs, bndFs, 'rows');
surf = surf(:, isBnd);

if direct_on
    VIZ.draw_mesh_3D(coord, surf);
    VIZ.draw_quantity_3D_old(coord, surf, zeros(size(coord)), pw, x1, x2, x3, y1, y2, z);
    VIZ.plot_displacements_3D(U2, coord, elem, surf, 0.05 * max(abs(coord(:))) / max(abs(U2(:))));
    VIZ.plot_deviatoric_strain_3D(U2, coord, elem, surf, B);

    figure; hold on; box on; grid on;
    plot(omega_hist2, lambda_hist2, '-o');
    title('Direct continuation method', 'Interpreter', 'latex');
    xlabel('variable - $\\xi$', 'Interpreter', 'latex');
    ylabel('strength reduction factor - $\\lambda$', 'Interpreter', 'latex');
end

if indirect_on
    VIZ.draw_mesh_3D(coord, surf);
    drawnow; pause(0.2);

    VIZ.plot_pore_pressure_3D(pw, coord, surf);
    drawnow; pause(0.2);

    VIZ.plot_displacements_3D(U3, coord, elem, surf, 0.05 * max(abs(coord(:))) / max(abs(U3(:))));
    drawnow; pause(0.2);

    VIZ.plot_deviatoric_strain_3D(U3, coord, elem, surf, B);
    cl = caxis(gca);
    caxis([cl(1), 0.25 * cl(2)]);
    drawnow; pause(0.2);

    plane_vals = {[], [35], [1e-16, 21.6506]};
    [figs, info] = VIZ.plot_deviatoric_norm_slices(B, U3, elem, coord, Xi, surf, plane_vals, 1); %#ok<NASGU,ASGLU>

    figure; hold on; box on; grid on;
    plot(omega_hist3, lambda_hist3, '-o');
    title('Indirect continuation method', 'Interpreter', 'latex');
    xlabel('control variable - $\\omega$', 'Interpreter', 'latex');
    ylabel('strength reduction factor - $\\lambda$', 'Interpreter', 'latex');
end

drawnow;
fprintf('Notebook workflow completed.\\n');